In [13]:
from pathlib import Path
import os

import pandas as pd
import numpy as np

from sqlalchemy import create_engine, text
import torch
import faiss
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel


BASE_DIR = Path.cwd().parent

ABSTRACTS_PATH = BASE_DIR / "data" / "arxiv_metadata.csv"

OUTPUT_EMBEDDINGS_PATH = BASE_DIR / "data" / "BERT_embeddings" / "embeddings.npy"
OUTPUT_INDEX_PATH = BASE_DIR / "data" / "BERT_embeddings" / "faiss_index.index"


In [3]:
abstracts_df = pd.read_csv(ABSTRACTS_PATH, usecols=["id", "paper_id", "abstract"])
print(f"Total papers to embed: {len(abstracts_df)}")
abstracts_df

Total papers to embed: 381344


/var/folders/kz/dlw3ql7s3ds1_72hwlw7_t540000gn/T/ipykernel_86542/486060465.py:1: DtypeWarning: Columns (0: paper_id) have mixed types. Specify dtype option on import or set low_memory=False.
  abstracts_df = pd.read_csv(ABSTRACTS_PATH, usecols=["id", "paper_id", "abstract"])


,id,paper_id,abstract
0,0,704.0009,We discuss the results from the combined IRA...
1,1,704.0017,Results from spectroscopic observations of t...
2,2,704.0023,"The very nature of the solar chromosphere, i..."
3,3,704.0044,We present a theoretical framework for plasm...
4,4,704.0048,We report on the analysis of selected single...
...,...,...,...
381339,381339,quant-ph/9903043,A non-local gauge symmetry of a complex scal...
381340,381340,quant-ph/9903053,The existence of a non-thermodynamic arrow o...
381341,381341,quant-ph/9907088,Fidelity plays a key role in quantum informa...
381342,381342,solv-int/9404002,Newtonian dynamical systems accepting the no...


#### See [https://www.sbert.net/docs/sentence_transformer/pretrained_models.html](https://www.sbert.net/docs/sentence_transformer/pretrained_models.html) and [https://huggingface.co/sentence-transformers/all-distilroberta-v1](https://huggingface.co/sentence-transformers/all-distilroberta-v1) for details about the pre-trained model used.

In [4]:
# load the pre-trained model and tokenizer
model_name = "sentence-transformers/all-distilroberta-v1"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # handle if run in CPU-only environment
model = model.to(device)
model.eval() # inference mode

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10189.95it/s]
RobertaModel LOAD REPORT from: sentence-transformers/all-distilroberta-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RobertaModel(
  (embeddings): RobertaEmbeddings(
    (word_embeddings): Embedding(50265, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
  )
  (encoder): RobertaEncoder(
    (layer): ModuleList(
      (0-5): 6 x RobertaLayer(
        (attention): RobertaAttention(
          (self): RobertaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): RobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (dropout)

In [5]:
# inspect token lengths for the embedding input
# this is important to determine the truncation strategy for long abstracts, since the model has a max token limit (512 for distilroberta)

texts = abstracts_df["abstract"].fillna("").tolist()

token_lengths = [
    len(tokenizer(text, add_special_tokens=True)["input_ids"])
    for text in texts
]

length_series = pd.Series(token_lengths)

print("Token length stats")
print(length_series.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

for cutoff in [64, 128, 256, 384, 512]:
    truncated = (length_series > cutoff).mean() * 100
    print(f"Would truncate at {cutoff:>3} tokens: {truncated:.2f}% of texts")


Token indices sequence length is longer than the specified maximum sequence length for this model (516 > 512). Running this sequence through the model will result in indexing errors


Token length stats
count    381344.000000
mean        292.799753
std         110.989292
min           5.000000
50%         294.000000
75%         376.000000
90%         435.000000
95%         468.000000
99%         538.000000
max         838.000000
dtype: float64
Would truncate at  64 tokens: 99.05% of texts
Would truncate at 128 tokens: 92.82% of texts
Would truncate at 256 tokens: 61.09% of texts
Would truncate at 384 tokens: 22.59% of texts
Would truncate at 512 tokens: 1.81% of texts


We'll use 512 tokens (the maximum length) for embedding.

In [6]:
def encode_texts(texts, batch_size=64):
    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]

        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True, # truncate to model's max length
            return_tensors="pt",
            max_length=512
        ).to(device)

        with torch.no_grad():
            output = model(**encoded)

        # mean pooling: average token embeddings
        token_embeddings = output.last_hidden_state # (batch, seq_len, 768) -> 768 is the embedding size for this model
        attention_mask = encoded.attention_mask # (batch, seq_len)

        # apply attention mask to get mean of valid tokens only
        mask_expanded = attention_mask.unsqueeze(-1).float()
        sum_embeddings = torch.sum(token_embeddings * mask_expanded, dim=1) # sum over seq_len
        sum_mask = mask_expanded.sum(dim=1).clamp(min=1e-9)
        mean_embeddings = sum_embeddings / sum_mask # (batch, 768)

        # L2 normalize
        #  this is for cosine similarity in FAISS since cosine similarity is just dot product of L2-normalized vectors
        normalized = F.normalize(mean_embeddings, p=2, dim=1)

        all_embeddings.append(normalized.cpu().numpy())

        if i % (batch_size * 10) == 0:
            print(f"Embedded {i}/{len(texts)} abstracts...")

    return np.concatenate(all_embeddings, axis=0)

In [8]:
# let's run the embedding! it may take a while
abstracts = abstracts_df["abstract"].tolist()
embeddings = encode_texts(abstracts, batch_size=50)
print(f"Generated embeddings shape: {embeddings.shape}")

np.save(OUTPUT_EMBEDDINGS_PATH, embeddings)

Embedded 0/381344 abstracts...
Embedded 500/381344 abstracts...
Embedded 1000/381344 abstracts...
Embedded 1500/381344 abstracts...
Embedded 2000/381344 abstracts...
Embedded 2500/381344 abstracts...
Embedded 3000/381344 abstracts...
Embedded 3500/381344 abstracts...
Embedded 4000/381344 abstracts...
Embedded 4500/381344 abstracts...
Embedded 5000/381344 abstracts...
Embedded 5500/381344 abstracts...
Embedded 6000/381344 abstracts...
Embedded 6500/381344 abstracts...
Embedded 7000/381344 abstracts...
Embedded 7500/381344 abstracts...
Embedded 8000/381344 abstracts...
Embedded 8500/381344 abstracts...
Embedded 9000/381344 abstracts...
Embedded 9500/381344 abstracts...
Embedded 10000/381344 abstracts...
Embedded 10500/381344 abstracts...
Embedded 11000/381344 abstracts...
Embedded 11500/381344 abstracts...
Embedded 12000/381344 abstracts...
Embedded 12500/381344 abstracts...
Embedded 13000/381344 abstracts...
Embedded 13500/381344 abstracts...
Embedded 14000/381344 abstracts...
Embedded 

In [9]:
embeddings = np.load(OUTPUT_EMBEDDINGS_PATH).astype(np.float32) # faiss requires float32
print(embeddings.shape)
print(np.linalg.norm(embeddings[0]))  # should be 1.0

(381344, 768)
1.0


In [11]:
dimension = embeddings.shape[1]
base_index = faiss.IndexFlatIP(dimension)
index = faiss.IndexIDMap(base_index) # map paper IDs with the vectors

faiss_ids = abstracts_df["id"].to_numpy(dtype=np.int64)
index.add_with_ids(embeddings.astype(np.float32), faiss_ids)

print(f"Total vectors indexed: {index.ntotal}")

# save index to disk
faiss.write_index(index, str(OUTPUT_INDEX_PATH))

Total vectors indexed: 381344


In [14]:
# read index for test

index = faiss.read_index(str(OUTPUT_INDEX_PATH))

In [15]:
# test performance

def embed(text):

    # make sure to use same configs as embedding
    model.eval()
    encoded = tokenizer(
        [text],
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        output = model(**encoded)

    token_embeddings = output.last_hidden_state
    attention_mask = encoded.attention_mask
    mask_expanded = attention_mask.unsqueeze(-1).float()
    sum_embeddings = torch.sum(token_embeddings * mask_expanded, dim=1)
    sum_mask = mask_expanded.sum(dim=1).clamp(min=1e-9)
    mean_embeddings = sum_embeddings / sum_mask
    query_vector = F.normalize(mean_embeddings, p=2, dim=1).cpu().numpy().astype("float32")

    return query_vector


def recommend(text: str, top_k: int = 50):

    query_vector = embed(text)
    scores, indices = index.search(query_vector, top_k)

    results = []
    for idx, score in zip(indices[0], scores[0]):
        row = df.iloc[idx]
        results.append({
            "title": row["title"],
            "categories": row["categories"],
            "update_year": row["update_year"],
            "score": round(float(score), 4),
        })
    return pd.DataFrame(results)

In [16]:
query = """
We investigate the rotational dynamics of Earth's liquid core using VLBI 
observations. Our analysis reveals two distinct nutation periods near -430 
solar days, suggesting a two-layer liquid core structure.
"""

results = recommend(query, top_k=10)
print(results.to_string())

: 